<a href="https://colab.research.google.com/github/AndrewUwU/EQ-Activiades-IARN/blob/main/Codigo_Vision_por_Red_neuronal_Caja_secreta_PIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Librerias
import cv2
import mediapipe as mp
import numpy as np
import urllib.request
import requests # Para enviar la señal de abrir al motor

In [ ]:
# ==========================================
# 1. CONFIGURACIÓN DE RED DEL ESP32-CAM
# ==========================================
#IP el Monitor Serie de Arduino hoy
IP_ESP32 = '172.20.10.7'
IP_SERVOS = '172.20.10.6'
URL_STREAM = f'http://{IP_ESP32}:81/stream'
URL_ABRIR = f'http://{IP_SERVOS}/activar'

In [ ]:
# ==========================================
# 2. INICIALIZACIÓN DE MEDIAPIPE (MANOS)
# ==========================================
mp_manos = mp.solutions.hands
mp_dibujo = mp.solutions.drawing_utils

detector_manos = mp_manos.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)


In [ ]:
# ==========================================
# 3. VARIABLES DE LA MÁQUINA DE ESTADOS
# ==========================================
frames_consecutivos = 0
LIMITE_FRAMES = 15  # Medio segundo aprox a 30 FPS
boveda_abierta = False

In [ ]:
# ==========================================
# 4. CONEXIÓN AL FLUJO DE VIDEO EN BRUTO
# ==========================================
print(f"Conectando al stream de video en {URL_STREAM}...")
try:
    stream = urllib.request.urlopen(URL_STREAM)
    bytes_data = bytes()
    print("¡Conexión exitosa! Iniciando escáner biométrico...")
    print("Presiona la tecla 'q' en la ventana de video para cerrar el programa.")
except Exception as e:
    print(f"Error fatal: No se pudo conectar al ESP32. Verifica que la IP sea {IP_ESP32} y que no tengas el stream abierto en el navegador.")
    print("Detalle del error:", e)
    exit()

In [ ]:
# ==========================================
# 5. CICLO PRINCIPAL (VISIÓN COMPUTACIONAL)
# ==========================================
while True:
    # 1. Descargar el flujo de datos por paquetes (chunks)
    bytes_data += stream.read(1024)

    # 2. Buscar marcadores Hexadecimales del inicio y fin de una foto JPEG
    a = bytes_data.find(b'\xff\xd8')
    b = bytes_data.find(b'\xff\xd9')

    # Si encontramos una imagen completa:
    if a != -1 and b != -1:
        # Extraer los bytes de la foto y limpiar el historial para la siguiente
        jpg = bytes_data[a:b+2]
        bytes_data = bytes_data[b+2:]

        # Convertir bytes a matriz OpenCV (Frame)
        frame = cv2.imdecode(np.frombuffer(jpg, dtype=np.uint8), cv2.IMREAD_COLOR)

        if frame is not None:
            # MediaPipe requiere RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Buscar manos en la imagen
            resultados = detector_manos.process(frame_rgb)

            # Si se detecta una mano en cámara:
            if resultados.multi_hand_landmarks:
                for mano_landmarks in resultados.multi_hand_landmarks:
                    # Dibujar las conexiones para comprobar que funciona
                    mp_dibujo.draw_landmarks(frame, mano_landmarks, mp_manos.HAND_CONNECTIONS)

                    # Incrementar el contador del estabilizador
                    frames_consecutivos += 1

                    # Si la mano se mantiene frente a la cámara el tiempo suficiente:
                    if frames_consecutivos >= LIMITE_FRAMES and not boveda_abierta:
                        print("¡Biometría validada! Enviando orden al servomotor...")
                        cv2.putText(frame, "ACCESO CONCEDIDO", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
                        cv2.imshow("Proyecto Boveda", frame)
                        cv2.waitKey(1) # Forzar refresco de la pantalla para que se vea el texto verde

                        try:
                            # Hacer que el cerebro (Laptop) envíe la orden al músculo (ESP32)
                            respuesta = requests.get(URL_ABRIR, timeout=5)
                            if respuesta.status_code == 200:
                                print("Motor activado con éxito.")
                            boveda_abierta = True
                        except requests.exceptions.RequestException as e:
                            print("Error de red al intentar abrir la bóveda:", e)

            else:
                # Si se quita la mano de la cámara, reiniciar el sistema
                frames_consecutivos = 0
                boveda_abierta = False

            # Interfaz gráfica de apoyo (HUD)
            if not boveda_abierta:
                texto_estado = f"Esperando mano... ({frames_consecutivos}/{LIMITE_FRAMES})"
                cv2.putText(frame, texto_estado, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            else:
                cv2.putText(frame, "BOVEDA ABIERTA", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            # Proyectar el video resultante
            cv2.imshow("Proyecto Boveda", frame)

            # Sistema de apagado seguro
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

In [ ]:
# Liberar memoria de Windows al terminar
cv2.destroyAllWindows()